# HW6 총액 regression model 을 고르고, API 로 내보낸다
이승민 · 24013735

입력: `sales_clean.csv` (HW5 단계④에서 만든 파일)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import platform
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

fonts = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}
plt.rcParams["font.family"] = fonts.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

## 단계① 읽고 나눈다: X = 수량 · 단가, y = 총액

In [2]:
sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## 단계② model 셋의 test R²

In [3]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])

lin  = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

print("linear regression        test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp   test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))
print("linear + qp coef :", lin2.coef_.round(3), " intercept :", round(lin2.intercept_, 1))

linear regression        test R² : 0.862
linear regression + qp   test R² : 1.0
decision tree (no limit) test R² : 0.994
linear + qp coef : [0. 0. 1.]  intercept : -0.0


## 단계③ 질문: linear regression 이 0.862 에 그친 이유, qp 를 더하면 1.0 이 되는 이유

**답** 총액은 수량 × 단가라는 곱셈인데, linear regression 은 `a·수량 + b·단가 + c` 처럼 각 열에 숫자를 곱해 더하는 것만 할 수 있어서 곱셈을 평평한 면으로 흉내 낼 수밖에 없다. 그래서 수량과 단가가 둘 다 크거나 둘 다 작은 주문에서 많이 틀리고 test R² 가 0.862 에 그친다.

qp = 수량 × 단가 열을 하나 더해 주면 곱셈이 이미 열 안에 들어가 있으므로, model 은 qp 의 계수를 1, 나머지를 0 으로 두기만 하면 된다(위 coef [0, 0, 1], 절편 0). 즉 「총액 = qp」를 찾아서 R² 가 1.0 이 된다.

## 단계④ max_depth 별 training · test R²

In [4]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## 단계⑤ cross-validation (cv=5) 으로 max_depth 고르기

In [5]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.3f}  std {s.std():.3f}")

max_depth 3     mean 0.845  std 0.019


max_depth 5     mean 0.961  std 0.003
max_depth 8     mean 0.992  std 0.000
max_depth 12    mean 0.995  std 0.000
max_depth None  mean 0.995  std 0.001


**결정** max_depth 8.

**이유** cross-validation 평균이 depth 8 에서 0.992, 12 와 제한 없음에서 0.995 로 0.003 차이밖에 나지 않고 표준편차도 셋 다 0.000 ~ 0.001 이다. score 가 거의 같으면 단순한 쪽을 고르므로 잎이 250 개인 depth 8 을 고른다(depth 12 는 잎 781 개).
training 1.000 과 test 0.994 의 차이가 작은 것은 총액 = 수량 × 단가라는 규칙이 실제로 있는 문제라서, 잘게 외워도 test 에서 그대로 맞기 때문이다.

## 단계⑥ 고른 model 을 저장 → API
API 에 올리는 model 은 linear regression + qp (단계⑦의 이유). 서버(`app.py`)에서도 학습 때와 같이 `qp` 열을 만들어 넣는다.

In [6]:
joblib.dump(lin2, "sales_model.pkl")

['sales_model.pkl']

In [7]:
# 수량 40 · 단가 200,000 에 대해 세 가지 답
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)
q = pd.DataFrame([[40, 200000.0]], columns=["quantity", "unit_price"])
q2 = q.assign(qp=q["quantity"] * q["unit_price"])
print("linear + qp       →", int(round(lin2.predict(q2)[0])))
print("tree (depth 8)    →", int(round(tree8.predict(q)[0])))
print("quantity × price  →", 40 * 200000)

linear + qp       → 8000000
tree (depth 8)    → 7362301
quantity × price  → 8000000


## 단계⑦ 왜 그 model 인가

**답** linear regression + qp 를 올렸다. 수량 40 · 단가 200,000 에 대해 linear + qp 는 8,000,000 으로 실제 곱 8,000,000 과 정확히 같고, tree (depth 8) 는 7,362,301 로 약 64만 원(8%) 틀린다.

tree 는 training 의 비슷한 주문들의 평균(계단)으로 답하므로 R² 0.99 라도 한 건 한 건은 어긋나고, training 범위 밖 값은 낼 수도 없다. 반면 linear + qp 는 공식(총액 = qp)을 그대로 찾았으므로 어떤 입력에도 곱을 정확히 돌려준다. 공식을 이미 알고 있으니 사실 model 없이 곱만 계산해도 되지만, model 을 API 로 올리는 연습이라면 공식과 똑같이 답하는 linear + qp 가 맞다.